# 2. Layer Heatmaps — EAP Score Visualization

Generates purple heatmaps of EAP attribution scores across layers and heads,
similar to Figure 2 in the MI_Bias paper.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

sns.set_theme(style='white')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

## Load Top Edges

In [ ]:
# Load edges for gender bias (run script 02 first)
import os

edges_path = '../results/top_edges_gender.json'

if not os.path.exists(edges_path):
    print(f'ERROR: {edges_path} not found!')
    print('Run first: python scripts/02_find_circuits.py --dataset data/gender_bias.json')
else:
    with open(edges_path, 'r') as f:
        edges = json.load(f)
    print(f'Loaded {len(edges)} edges')
    print(f'Top 5 edges:')
    for e in edges[:5]:
        print(f'  L{e["src_layer"]}.{e["src_type"]}', end='')
        if e['src_head'] is not None:
            print(f'.H{e["src_head"]}', end='')
        print(f' -> L{e["dst_layer"]}.{e["dst_type"]}', end='')
        if e['dst_head'] is not None:
            print(f'.H{e["dst_head"]}', end='')
        print(f' (score={e["score"]:.6f})')

## Attention Head Heatmap (Source Contribution)

Shows which attention heads (across layers) have the highest cumulative EAP
scores as **sources** of bias-causing information flow.

In [ ]:
# Determine model dimensions from edges
n_layers = max(max(e['src_layer'], e['dst_layer']) for e in edges) + 1
n_heads = max(e['src_head'] for e in edges if e['src_head'] is not None) + 1

print(f'Model: {n_layers} layers, {n_heads} heads per layer')

# Build source heatmap: accumulate scores per (layer, head)
src_heatmap = np.zeros((n_layers, n_heads))
for e in edges:
    if e['src_type'] == 'attn' and e['src_head'] is not None:
        src_heatmap[e['src_layer'], e['src_head']] += e['score']

# Custom purple colormap (like Fig 2)
purple_cmap = LinearSegmentedColormap.from_list('purple_bias', [
    '#1a0a2e',  # Very dark purple
    '#3d1a78',  # Dark purple
    '#7b2fbe',  # Medium purple
    '#b266ff',  # Light purple
    '#e0b3ff',  # Very light purple
    '#ffffff',  # White
][::-1])

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(src_heatmap, cmap=purple_cmap, aspect='auto', interpolation='nearest')
ax.set_xlabel('Attention Head', fontsize=13)
ax.set_ylabel('Layer', fontsize=13)
ax.set_title('EAP Source Attribution Scores (Attention Heads)', fontsize=15, fontweight='bold')
ax.set_xticks(range(n_heads))
ax.set_yticks(range(n_layers))

# Add score annotations
for i in range(n_layers):
    for j in range(n_heads):
        val = src_heatmap[i, j]
        if val > 0:
            text_color = 'white' if val > src_heatmap.max() * 0.5 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=7, color=text_color)

plt.colorbar(im, ax=ax, label='Cumulative EAP Score', shrink=0.8)
plt.tight_layout()
plt.savefig('../results/figures/eap_source_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## Destination Heatmap

Shows which heads **receive** the most bias-causing information.

In [ ]:
# Build destination heatmap
dst_heatmap = np.zeros((n_layers, n_heads))
for e in edges:
    if e['dst_type'] == 'attn' and e['dst_head'] is not None:
        dst_heatmap[e['dst_layer'], e['dst_head']] += e['score']

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(dst_heatmap, cmap=purple_cmap, aspect='auto', interpolation='nearest')
ax.set_xlabel('Attention Head', fontsize=13)
ax.set_ylabel('Layer', fontsize=13)
ax.set_title('EAP Destination Attribution Scores (Attention Heads)', fontsize=15, fontweight='bold')
ax.set_xticks(range(n_heads))
ax.set_yticks(range(n_layers))

for i in range(n_layers):
    for j in range(n_heads):
        val = dst_heatmap[i, j]
        if val > 0:
            text_color = 'white' if val > dst_heatmap.max() * 0.5 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=7, color=text_color)

plt.colorbar(im, ax=ax, label='Cumulative EAP Score', shrink=0.8)
plt.tight_layout()
plt.savefig('../results/figures/eap_destination_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## MLP Contribution by Layer

In [ ]:
# MLP source scores by layer
mlp_src_scores = np.zeros(n_layers)
mlp_dst_scores = np.zeros(n_layers)

for e in edges:
    if e['src_type'] == 'mlp':
        mlp_src_scores[e['src_layer']] += e['score']
    if e['dst_type'] == 'mlp':
        mlp_dst_scores[e['dst_layer']] += e['score']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(n_layers), mlp_src_scores, color='#7b2fbe', alpha=0.85)
axes[0].set_yticks(range(n_layers))
axes[0].set_ylabel('Layer')
axes[0].set_xlabel('Cumulative EAP Score')
axes[0].set_title('MLP as Source of Bias', fontsize=14, fontweight='bold')

axes[1].barh(range(n_layers), mlp_dst_scores, color='#b266ff', alpha=0.85)
axes[1].set_yticks(range(n_layers))
axes[1].set_ylabel('Layer')
axes[1].set_xlabel('Cumulative EAP Score')
axes[1].set_title('MLP as Destination of Bias', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/eap_mlp_contribution.png', dpi=200, bbox_inches='tight')
plt.show()

## Edge Flow Diagram (Top 20 Edges)

In [ ]:
# Visualize the top 20 edges as a connectivity diagram
top_n = 20
top_edges = edges[:top_n]

fig, ax = plt.subplots(figsize=(16, 8))

for i, e in enumerate(top_edges):
    src_y = e['src_layer']
    dst_y = e['dst_layer']
    src_x = e['src_head'] if e['src_head'] is not None else -1
    dst_x = e['dst_head'] if e['dst_head'] is not None else -1
    
    alpha = 0.3 + 0.7 * (e['score'] / top_edges[0]['score'])
    width = 1 + 3 * (e['score'] / top_edges[0]['score'])
    
    ax.annotate('', xy=(dst_x, dst_y), xytext=(src_x, src_y),
                arrowprops=dict(arrowstyle='->', color='#7b2fbe',
                               lw=width, alpha=alpha))

ax.set_xlabel('Head Index (-1 = MLP)', fontsize=13)
ax.set_ylabel('Layer', fontsize=13)
ax.set_title(f'Top {top_n} Bias-Causing Edge Connections', fontsize=15, fontweight='bold')
ax.set_xlim(-2, n_heads)
ax.set_ylim(-0.5, n_layers - 0.5)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../results/figures/edge_flow_diagram.png', dpi=200, bbox_inches='tight')
plt.show()